### QQP with Mamba

In [ ]:
from __future__ import annotations
from typing import Dict, Any

import torch
from pathlib import Path

from qqp_task import QQPTask
from src.train_utils.trainer import Trainer

### Configuration
We define a set of hyperparameters and configurations for the training process.
This includes data paths, batch sizes, learning rates, and model-specific parameters.
Using unified block factory from src.utils.block_factory for consistent configuration.

In [ ]:
from src.utils.block_factory import make_mamba_block_cfg_ctor

In [ ]:
current_dir = Path.cwd()
project_root = current_dir.parent.parent.parent
data_root = str(project_root / "src" / "datasets" / "qqp" / "data")

args: Dict[str, Any] = {
    "data_root": data_root,
    "batch": 256,
    "data_loader_kwargs": {
        "num_workers": 0,
        "pin_memory": False,
        "persistent_workers": False,
        # QQP-specific parameters
        "seed": 42,
        "val_ratio": 0.1,
        "test_ratio": 0.1,
        "lowercase": True,
        "max_len": 64,
        "max_vocab": 50_000,
        "min_freq": 2,
    },

    "epochs": 100,
    "lr": 1e-3,
    "wd": 1e-4,
    "amp": False,
    "save_dir": "./runs/qqp_mamba_task",
    "warmup_epochs": 5,
    "patience": 5,
    "min_delta": 0.001,
    "early_key": "accuracy",

    "d_model": 128,
    "depth": 1,
    "dropout": 0.1,
    "mlp_ratio": 2.0,
    "droppath_final": 0.0,
    "layerscale_init": 0.0,
    "residual_gain": 1.0,
    "pool": "mean",

    # Mamba-specific parameters
    "d_state": 16,
    "expand_factor": 2,
    "d_conv": 4,
    "use_external_mlp": False,

    # QQP specific
    "max_len": 64,
}

args["block_cfg_ctor"] = make_mamba_block_cfg_ctor(
    d_state=args["d_state"],
    expand_factor=args["expand_factor"],
    d_conv=args["d_conv"],
    use_external_mlp=args["use_external_mlp"],
    dropout=args["dropout"],
    mlp_ratio=args["mlp_ratio"],
    droppath_final=args["droppath_final"],
    layerscale_init=args["layerscale_init"],
    residual_gain=args["residual_gain"],
    pool=args["pool"],
)

if torch.cuda.is_available():
    args["device"] = torch.device("cuda")
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    args["device"] = torch.device("mps")
    print("Using MPS (Apple Silicon)")
else:
    args["device"] = torch.device("cpu")
    args["amp"] = False
    print("Using CPU")

### Training
With the configuration set up, we can now instantiate the `QQPTask` and the `Trainer`.
The `fit` method on the trainer will start the training process, which includes training,
validation, and saving the best model based on the validation accuracy.

In [ ]:
from src.utils.visualization import plot_classification_history as plot_history

In [ ]:
task = QQPTask()

if torch.backends.mps.is_available() and args["device"].type == "mps":
    torch.mps.set_per_process_memory_fraction(0.9)

trainer = Trainer(args=args, task=task)

best_metric, ckpt_path = trainer.fit()
print(f"Done. Best {trainer.early_key}={best_metric:.4f} @ {ckpt_path}")

In [ ]:
from src.utils.checkpoint import load_trainer_from_checkpoint

trainer = load_trainer_from_checkpoint(
    checkpoint_path=args["save_dir"] + "/best.pt",
    args=args,
    task=QQPTask(),
)

history = trainer.history

plot_history(history, model_name="Mamba")

### Evaluation
After training, we can evaluate the best model on the test set.
We load the best model from the checkpoint and then run the evaluation.
The results, including accuracy, are printed.

In [ ]:
from src.eval.eval_utils import evaluate_classification_model as evaluate_best_model

logits_test, labels_test = evaluate_best_model(
    args=args,
    task=QQPTask(),
    best_model_path=f"{args['save_dir']}/best.pt",
    num_classes=2,
    use_test_set=True,
)

### Changes in dataset length

In [ ]:
from src.eval.eval_utils import evaluate_classification_model as evaluate_best_model

if torch.backends.mps.is_available() and args["device"].type == "mps":
    torch.mps.set_per_process_memory_fraction(0.9)


for frac in [0.1, 0.25, 0.5]:
    print(f"\nTraining with fraction: {frac}")

    args["fraction"] = frac
    args["save_dir"] = f"./runs/qqp_mamba_task_frac_{int(frac*100)}"
    trainer = Trainer(args=args, task=QQPTask())
    best_metric, best_path = trainer.fit()

    print(f"\nTraining complete for fraction {frac}! Best validation {trainer.early_key}: {best_metric:.4f}")
    print(f"Best model saved to: {best_path}")

    history = trainer.history

    plot_history(history, model_name="Mamba")

    logits_test, labels_test = evaluate_best_model(
        args=args,
        task=QQPTask(),
        best_model_path=best_path,
        num_classes=2,
        use_test_set=True,
        additional_metrics=['precision', 'recall', 'pr_auc', 'roc_auc', 'f1_score']
    )
